# The 3D Visualizer

The existing {doc}`Visualizer <using-the-visualizer>` draws a liquid handler's deck. This one
draws a cartesian space and whatever is placed in it: machines, benches, and anything that
moves between them. It reads the resource tree and the tracking state each resource publishes,
so nothing about a particular machine is written into the viewer.
## Before you start

- **No hardware is needed.** The STAR here runs in simulation.
- **A browser with WebGPU**, which is Chrome, Edge and Safari 18 or newer. Firefox and older
  browsers fall back to WebGL2 and render the same scene; only GIF recording is unavailable there.
- **Nothing extra to install.** The viewer ships with PyLabRobot and runs offline.

This notebook uses `await` at the top level, which Jupyter supports directly.


## Building a facility

A **`Facility`** is the root: the space everything stands in, with its own origin. A machine is
placed in it at a coordinate, the same way a plate is placed on a carrier. Anything else that
stands on the floor goes in the same way, machine or not.

The facility is built first and handed to the viewer empty. Everything after that is added while
the viewer is watching: a resource assigned anywhere in the tree tells its parent, which tells its
own parent, so the viewer hears about it wherever it lands and rebuilds the scene.

In [1]:
from pylabrobot.hamilton.star.device import STAR
from pylabrobot.resources import set_tip_tracking, set_volume_tracking
from pylabrobot.resources.coordinate import Coordinate
from pylabrobot.resources.corning import cor_96_wellplate_360uL_Fb
from pylabrobot.resources.hamilton import PLT_CAR_L5AC_A00, TIP_CAR_480_A00
from pylabrobot.resources.hamilton import hamilton_96_tiprack_1000uL
from pylabrobot.resources.resource import Resource
from pylabrobot.visualizer3D import Facility, Viewer3D

# Tracking is what the viewer reads to colour wells and show tips, so turn it on.
set_volume_tracking(True)
set_tip_tracking(True)


In [2]:
facility = Facility(name="facility", size_x=2800, size_y=2700, size_z=2400)
facility

Facility(name='facility', location=Coordinate(000.000, 000.000, 000.000), size_x=2800, size_y=2700, size_z=2400, category=facility)

## Starting the viewer

This serves the page and opens a browser tab, on a facility with nothing in it yet. The viewer
stays connected from here on, so every cell below adds something you can watch appear.

In [3]:
# Where the models live. Every `reference_glb` is relative to this. Prototyping path: point it at
# your own directory before this notebook goes anywhere public.
MODELS_ROOT = "/Users/camillomoschner/Documents/GitHub/bct_animations/2608_STAR/web_export"

viewer = Viewer3D(
  facility,
  name="3d-visualizer.ipynb",
  models_root=MODELS_ROOT,
)
await viewer.start()

viewer on http://127.0.0.1:1338  (websocket 2122)


### Something that is not a machine

A bench has no deck and no driver, and still takes part in the same space. This is the case the
deck-shaped viewer had no way to express. It is also what the STARlet stands on here.

In [4]:
star_bench = Resource(name="star_bench", size_x=1900, size_y=900, size_z=920, category="bench")
facility.assign_child_resource(star_bench, location=Coordinate(0, 1800, 0))


pfbench = Resource(name="pf_bench", size_x=900, size_y=900, size_z=920, category="bench")
facility.assign_child_resource(pfbench, location=Coordinate(1900, 1800, 0))


starlet_bench = Resource(name="starlet_bench", size_x=1300, size_y=900, size_z=920, category="bench")
starlet_bench.rotate(z=-90)
facility.assign_child_resource(starlet_bench, location=Coordinate(1900, 1800, 0))


### A machine in the facility

`STAR(simulation=True)` gives a STAR that answers as the real instrument would, so the deck,
the channels and the X-arm all come from the machine's own configuration rather than from
anything written here. `extension_housing=True` fits the cabinet that stands to its left.

Building it does not place it. It goes on the bench a few cells down, once it has been asked what
it says about itself.

In [5]:
star = STAR(
    simulation=True,
    extension_housing=True
    )
await star.setup()

2026-08-28 17:05:57,345 - pylabrobot.hamilton.star.driver.master - WARNING - the head96 reports itself uninitialized, and there is nowhere configured to eject at. Set head96.configuration.tip_discard_location, or pass it to head96.initialize().
2026-08-28 17:05:57,347 - pylabrobot.resources.hamilton.hamilton_decks - WARNING - Resource 'pipette_channel_0' is very high on the deck: 474.7 mm. Be careful when traversing the deck.
2026-08-28 17:05:57,358 - pylabrobot.resources.hamilton.hamilton_decks - WARNING - Resource 'pipette_channel_0' is very high on the deck: 474.7 mm. Be careful when grabbing this resource.
2026-08-28 17:05:57,388 - pylabrobot.resources.hamilton.hamilton_decks - WARNING - Resource 'pipette_channel_0_tip_mounting_shaft' is very high on the deck: 334.7 mm. Be careful when traversing the deck.
2026-08-28 17:05:57,389 - pylabrobot.resources.hamilton.hamilton_decks - WARNING - Resource 'pipette_channel_0_tip_mounting_shaft' is very high on the deck: 334.7 mm. Be careful 

### Two declarations the machine makes about itself

The viewer holds no constants about any instrument. Where it needs to draw something specific -
how far the channels reach across the deck, and the opening through the X-arm's carriage - the
resource declares it and the viewer draws whatever it is told.

The deck height below is a local override. A STAR deck reports `size_z` of 900 mm, which is the
working envelope rather than the deck's own extent. The honest height is the top of the arm that
rides above it. Applied here rather than in the resource library, pending a fix upstream.


In [6]:
from pylabrobot.visualizer3D.demo import declare_channel_access

declare_channel_access(star)

DECK_HEIGHT = 334.7 + 140.0  # channel travel, plus the arm's own height
star.deck._size_z = DECK_HEIGHT
star.deck._local_size_z = DECK_HEIGHT


### Drawing a resource from its own 3D file

A resource is drawn as its bounding box, because a box is what its size describes. A resource can
instead name a **`reference_glb`** - a path to a glTF binary, relative to a model root the viewer is
given - and the viewer draws that in its place. Anything without one keeps its box.

The file is authored in that resource's OWN frame, in metres, Z up. There is nothing else to
declare: no units, no up-axis, no offset. A viewer places the model by the resource's own transform,
which is the same transform it would have used for the box.

That is the whole mechanism, and it is why this generalises past one machine. The instrument names
its chassis, the deck names whichever of its two front panels applies, the autoload's tray and sled
name theirs, and each is placed by the resource that owns it. Nothing walks a manifest, and nothing
in the viewer knows what a STAR is.

`reference_glb` is inert outside a viewer: it is a string on the resource that nothing opens, looks
for or validates, so a headless or simulated run is unaffected by it.

In [7]:
star.reference_glb = "star_base.glb"

# The deck's front is one of two, never both: a machine with an autoload has the belt running
# where a manual one has a cover plate over the bay. This machine reports an autoload, so it takes
# the belt frame - `star_deck_front_top_cover.glb` is the other.
star.deck.reference_glb = "star_autoload_tray_belt_frame.glb"

# The machine's left side is the same kind of pair. An extension housing has no machine-facing
# side, so a machine that has one has no left side panel: whichever resource exists takes its own
# model, and the other name simply is not in the tree.
by_name = {r.name: r for r in star.get_all_children()}
for resource_name, model in (
  ("autoload_loading_tray", "star_autoload_tray.glb"),
  ("autoload_sled", "star_autoload_sled.glb"),
  ("left_side_panel", "star_side_panel_left.glb"),
  ("left_extension_housing", "left_extension_housing.glb"),
):
  resource = by_name.get(resource_name)
  if resource is not None:
    resource.reference_glb = model

[(r.name, r.reference_glb) for r in [star, star.deck] + list(by_name.values()) if r.reference_glb]

[('Hamilton STAR', 'star_base.glb'),
 ('deck', 'star_autoload_tray_belt_frame.glb'),
 ('deck', 'star_autoload_tray_belt_frame.glb'),
 ('left_extension_housing', 'left_extension_housing.glb'),
 ('autoload_sled', 'star_autoload_sled.glb'),
 ('autoload_loading_tray', 'star_autoload_tray.glb')]

### Putting it on the bench

Everything above was said to a machine standing outside the facility, where the viewer could not
see it. Assigning it is what the viewer hears, and a scene is rebuilt on a tree change rather than
on an attribute being set - so the declarations above are already in place when the machine is
first drawn.

In [8]:
star_bench.assign_child_resource(star, location=Coordinate(200, 100, star_bench.get_size_z()))

### Loading the deck

Carriers go on rails, and their racks and plates go in slots. Each assignment reaches the viewer
by the path the machine took.

In [9]:
tip_carrier = TIP_CAR_480_A00(name="tip_carrier")
for slot in range(3):
  tip_carrier[slot] = hamilton_96_tiprack_1000uL(name=f"tips_{slot}")
star.deck.assign_child_resource(tip_carrier, rails=1)

source_carrier = PLT_CAR_L5AC_A00(name="source_carrier")
for slot in range(5):
  source_carrier[slot] = cor_96_wellplate_360uL_Fb(name=f"source_{slot}")
star.deck.assign_child_resource(source_carrier, rails=8)

destination_carrier = PLT_CAR_L5AC_A00(name="destination_carrier")
for slot in range(5):
  destination_carrier[slot] = cor_96_wellplate_360uL_Fb(name=f"destination_{slot}")
star.deck.assign_child_resource(destination_carrier, rails=14)


## Watching state change

Everything from here changes tracking state, and the viewer follows without being told to
redraw. Wells take colour from how full they are, and tip spots show whether a tip is fitted.


In [10]:
source = star.deck.get_resource("source_0")
destination = star.deck.get_resource("destination_0")
tips = star.deck.get_resource("tips_0")

for well in source.get_all_items():
  well.tracker.set_volume(300.0)


Taking eight tips out of the rack. Watch the first column of `tips_0` empty.


In [11]:
import asyncio

for row in "ABCDEFGH":
  spot = tips.get_item(f"{row}1")
  if spot.tracker.has_tip:
    spot.tracker.remove_tip()
  await asyncio.sleep(0.1)


Moving liquid from one plate to the other, a column at a time. Only wells that actually hold
something are moved, so no channel is asked to transfer nothing.


In [12]:
for column in range(1, 13):
  for row in "ABCDEFGH":
    well = f"{row}{column}"
    taken = min(150.0, source.get_item(well).tracker.get_used_volume())
    if taken <= 0:
      continue
    source.get_item(well).tracker.remove_liquid(taken)
    destination.get_item(well).tracker.add_liquid(volume=taken)
  await asyncio.sleep(0.3)


## The X-arm

The arm is a resource on the deck like any other, and it moves. Its carriage is drawn
see-through with an outline, so you can read the deck underneath it, and the cyan line marks its
reference point in x.

Where it is arrives the same way a well's volume does. `move_x` reads the position back off the
machine and writes it to the arm resource, and setting a resource's location publishes it - so the
picture follows a measurement, on the one channel everything else uses. Nothing polls the machine
to draw this.

In [21]:
for x in (200.0, 900.0, 600.0):
  await star.x_arm.move_x(x)
  await asyncio.sleep(1.6)


In [23]:
await star.autoload.move_x(200.0)
await star.autoload.park()

In [25]:
await star.head96.move_z(290.0)

## A rigged mesh

The STARlet above is a single static file. A rigged file goes further: it names the parts that
move, so one mesh can follow a machine's joints instead of standing still. The PF400 below is
that case.

In [16]:
# The rigged export this was developed against. Prototyping path: swap it for your own file
# before this notebook goes anywhere public.
PF400_GLB = "/Users/camillomoschner/Documents/GitHub/bct_animations/2606_pf400_basic_movement/web_export/pf400.glb"


### Joints, and why the viewer knows nothing about arms

A rigged file names the parts that move. The declaration maps each joint to the node that answers
to it, which axis it acts about, and whether it turns (`revolute`) or slides (`prismatic`). The
viewer moves what it is told and holds no geometry for any particular arm.

The keys are PyLabRobot's own `Axis` values for the PreciseFlex, so the same numbers that command
the arm also drive the picture. On a PF400 `Axis.BASE` is the vertical column, and the three
rotations that follow make it a SCARA.


In [17]:
from pylabrobot.brooks.precise_flex import Axis

JOINTS = {
  str(int(Axis.BASE)): {"node": "J1_Z", "axis": "z", "type": "prismatic"},
  str(int(Axis.SHOULDER)): {"node": "J2_Shoulder", "axis": "z", "type": "revolute"},
  str(int(Axis.ELBOW)): {"node": "J3_Elbow", "axis": "z", "type": "revolute"},
  str(int(Axis.WRIST)): {"node": "J4_Wrist", "axis": "z", "type": "revolute"},
}


### A resource that publishes joint angles

The viewer follows tracking state, so an arm that wants its picture to move has to publish where
its joints are. `PreciseFlex` is a driver rather than a resource today and publishes nothing, so
this small resource stands in for that: it holds the pose and tells anyone listening when it
changes. The same shape is what the driver would need to grow.


In [18]:
from typing import Any, Dict


class ArmResource(Resource):
  """A resource that publishes joint angles, so a rigged mesh can follow them."""

  def __init__(self, name, size_x, size_y, size_z, category="arm"):
    super().__init__(name=name, size_x=size_x, size_y=size_y, size_z=size_z, category=category)
    self.joints: Dict[str, float] = {}

  def serialize_state(self) -> Dict[str, Any]:
    return {**super().serialize_state(), "joints": dict(self.joints)}

  def set_joints(self, pose: Dict[Axis, float]) -> None:
    """Angles in degrees, travel in mm, keyed by axis."""
    self.joints.update({str(int(axis)): float(value) for axis, value in pose.items()})
    self._state_updated()


In [19]:
# The arm's own extents. Measured from the file: 484 x 235 x 679 mm.
pf400 = ArmResource(name="pf400", size_x=484, size_y=235, size_z=679)
pf400.mesh = {"path": PF400_GLB, "units": "cm", "up": "Z", "joints": JOINTS}
pf400.rotate(z=-90)
pfbench.assign_child_resource(pf400, location=Coordinate(300, 500, pfbench.get_size_z()))


Now move it. Each pose is published as state, and the mesh follows without the viewer being told
to redraw - the same path the wells and tips took earlier.


In [26]:
for pose in (
  {Axis.BASE: 0.0, Axis.SHOULDER: 0.0, Axis.ELBOW: 0.0, Axis.WRIST: 0.0},
  {Axis.BASE: 200.0, Axis.SHOULDER: 45.0, Axis.ELBOW: -60.0, Axis.WRIST: 20.0},
  {Axis.BASE: 350.0, Axis.SHOULDER: -30.0, Axis.ELBOW: 90.0, Axis.WRIST: -40.0},
):
  pf400.set_joints(pose)
  await asyncio.sleep(2)


Two things worth knowing about the file itself.

**Scale is not declared by glTF**, so it has to be established. Here it was cross-checked against
`pylabrobot.brooks.precise_flex.kinematics.ARM_LINKS_STANDARD`, which gives the shoulder-to-elbow
and elbow-to-wrist lengths as 225 mm and 210 mm. The file's joint origins sit 23.6 and 21.2 units
apart, so one unit is ten millimetres and this is the standard-reach arm rather than the extended
one. Node origins are not exactly kinematic centres, so expect a few percent.

**Compressed meshes need a decoder.** Files exported for the web are often Draco-compressed. The
decoder is vendored and fetched only when a compressed file actually turns up, so a viewer that
never loads one pays nothing for it.


## Stopping

The page stays open and reports itself disconnected. Re-running the start cell reconnects it.


In [42]:
await viewer.stop()
